In [ ]:
# Cell 0: Install required library for Azure Data Lake Storage upload
!pip install azure-storage-blob faker

In [ ]:
# Cell 1: Imports and configuration
import uuid
import random
from datetime import datetime, timedelta
import xml.etree.ElementTree as ET
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient
from faker import Faker
import io
import time
import threading

In [ ]:
# -- Configurable Variables --
BLOB_CONNECTION_STRING = "TBD"
CONTAINER_NAME = "rawshippingmsgs"
SHIPPING_PROVIDERS = ["FedEx", "DPD", "UPS"]
DISTRIBUTION_CENTERS = [
    {"country": "Germany", "city": "Berlin", "lat": 52.5200, "long": 13.4050},
    {"country": "Sweden", "city": "Stockholm", "lat": 59.3293, "long": 18.0686},
    {"country": "Estonia", "city": "Tallinn", "lat": 59.4370, "long": 24.7536}
]
ZAVA_PROD_CATEGORIES = ["GenZ Pros", "Altars", "Colours", "Kids"]
ZAVA_PROD_NAMES = [
    "Berlin Classic", "Stockholm Modern", "Tallinn Kids", 
    "Vivid Runner", "Rainbow Dream", "Unicorn Spark", 
    "Midnight Black", "Emerald Green", "Sunshine Yellow"
]

# European destination countries with major cities and coordinates
DESTINATIONS = [
    {"country": "Germany", "city": "Frankfurt", "lat": 50.1109, "long": 8.6821},
    {"country": "France", "city": "Paris", "lat": 48.8566, "long": 2.3522},
    {"country": "Netherlands", "city": "Amsterdam", "lat": 52.3676, "long": 4.9041},
    {"country": "Belgium", "city": "Brussels", "lat": 50.8503, "long": 4.3517},
    {"country": "Italy", "city": "Rome", "lat": 41.9028, "long": 12.4964},
    {"country": "Spain", "city": "Madrid", "lat": 40.4168, "long": -3.7038},
    {"country": "Sweden", "city": "Gothenburg", "lat": 57.7089, "long": 11.9746},
    {"country": "Denmark", "city": "Copenhagen", "lat": 55.6761, "long": 12.5683},
    {"country": "Finland", "city": "Helsinki", "lat": 60.1699, "long": 24.9384},
    {"country": "Poland", "city": "Warsaw", "lat": 52.2297, "long": 21.0122},
    {"country": "Austria", "city": "Vienna", "lat": 48.2082, "long": 16.3738},
    {"country": "Czech Republic", "city": "Prague", "lat": 50.0755, "long": 14.4378},
    {"country": "Hungary", "city": "Budapest", "lat": 47.4979, "long": 19.0402},
    {"country": "Estonia", "city": "Tartu", "lat": 58.3776, "long": 26.7290},
    {"country": "Norway", "city": "Oslo", "lat": 59.9139, "long": 10.7522}
]

In [ ]:
faker = Faker()

# Core helper functions: faker-based realistic address and products generation

def random_address(destination):
    streets = ["Main St", "High St", "Elm St", "Oak St", "Maple Ave"]
    zip_code = str(random.randint(10000, 99999))
    street = random.choice(streets)
    number = str(random.randint(1, 200))
    name = faker.name()
    return {
        "name": name,
        "street": f"{street} {number}",
        "city": destination["city"],
        "zip": zip_code,
        "country": destination["country"],
        "latitude": destination["lat"],
        "longitude": destination["long"]
    }

def random_shipping_contents():
    count = random.randint(1, 3)
    return [{
        "category": random.choice(ZAVA_PROD_CATEGORIES),
        "name": random.choice(ZAVA_PROD_NAMES),
        "size": random.choice(["36", "38", "40", "42", "44"]),
        "qty": random.randint(1, 2)
    } for _ in range(count)]

def random_weight(items):
    return round(sum(random.uniform(0.7, 1.2) * i['qty'] for i in items), 2)

def random_provider(): return random.choice(SHIPPING_PROVIDERS)
def random_distribution_center(): return random.choice(DISTRIBUTION_CENTERS)
def random_destination(): return random.choice(DESTINATIONS)

# Generate timestamps with possible delays & failures injected

def generate_event_time(start_time, min_hours=0, max_hours=72, failure_delay=False):
    delay = random.uniform(min_hours, max_hours)
    if failure_delay:
        delay += random.uniform(24, 168)  # 1 to 7 days more delay on failure
    ts = start_time + timedelta(hours=delay)
    return ts.replace(
        minute=random.randint(0,59), second=random.randint(0,59), microsecond=random.randint(0,999999)
    )

# Build XML for shipment events

def create_shipping_xml_event(order_number, src, dst, items, weight, provider, status, event_time, failure_note=None, metadata=None):
    root = ET.Element("ShippingEvent")
    ET.SubElement(root, "Provider").text = provider
    ET.SubElement(root, "OrderNumber").text = order_number
    ET.SubElement(root, "EventTime").text = event_time.isoformat() + "Z"
    ET.SubElement(root, "Status").text = status
    ET.SubElement(root, "Timestamp").text = datetime.now().isoformat() + "Z"

    dc = ET.SubElement(root, "SourceDistributionCenter")
    ET.SubElement(dc, "City").text = src["city"]
    ET.SubElement(dc, "Country").text = src["country"]
    ET.SubElement(dc, "Latitude").text = str(src["lat"])
    ET.SubElement(dc, "Longitude").text = str(src["long"])

    addr = ET.SubElement(root, "DestinationAddress")
    ET.SubElement(addr, "Name").text = dst["name"]
    ET.SubElement(addr, "Street").text = dst["street"]
    ET.SubElement(addr, "City").text = dst["city"]
    ET.SubElement(addr, "Zip").text = dst["zip"]
    ET.SubElement(addr, "Country").text = dst["country"]
    ET.SubElement(addr, "Latitude").text = str(dst["latitude"])
    ET.SubElement(addr, "Longitude").text = str(dst["longitude"])

    ship_contents = ET.SubElement(root, "ShippingContents")
    for item in items:
        prod = ET.SubElement(ship_contents, "Product")
        ET.SubElement(prod, "Category").text = item["category"]
        ET.SubElement(prod, "Name").text = item["name"]
        ET.SubElement(prod, "Size").text = item["size"]
        ET.SubElement(prod, "Quantity").text = str(item["qty"])

    ET.SubElement(root, "WeightKg").text = str(weight)

    if failure_note:
        ET.SubElement(root, "FailureNote").text = failure_note
        
    # Optional metadata for extended dashboarding info
    if metadata:
        meta_elem = ET.SubElement(root, "Metadata")
        for k, v in metadata.items():
            ET.SubElement(meta_elem, k).text = str(v)

    return ET.tostring(root, encoding='utf-8')

def upload_xml_to_blob(xml_bytes, filename):
    blob_client = container_client.get_blob_client(filename)
    blob_client.upload_blob(io.BytesIO(xml_bytes), overwrite=True)
    # print(f"Uploaded: {filename}")

# Main generator for multi-stage, large volume shipping events with realistic scenarios

def generate_large_scale_shipping_events(total_orders=1000, max_events_per_second=5):
    base_date = datetime.utcnow() - timedelta(days=30)
    events_emitted = 0
    event_interval = 1.0 / max_events_per_second

    for order_index in range(total_orders):
        order_number = str(uuid.uuid4())
        src = random_distribution_center()
        dst_info = random_destination()
        dst = random_address(dst_info)
        items = random_shipping_contents()
        weight = random_weight(items)
        provider = random_provider()
        timestamp = datetime.now()

        dispatch_time = base_date + timedelta(
            days=random.uniform(0, 30),
            hours=random.uniform(0, 23),
            minutes=random.uniform(0, 59),
            seconds=random.uniform(0, 59)
        )
        
        # Dispatch event
        dispatched_xml = create_shipping_xml_event(
            order_number, src, dst, items, weight, provider,
            "Dispatched", dispatch_time
        )
        upload_xml_to_blob(dispatched_xml, f"{provider}_order_{order_number}_dispatched.xml")
        events_emitted += 1
        time.sleep(event_interval)

        # Simulate lifecycle events with probabilities

        failure_flags = {
            "delay_pickup": random.random() < 0.15,
            "lost_pickup": random.random() < 0.03,
            "delay_routing": random.random() < 0.10,
            "reroute": random.random() < 0.05,
            "cancelled": random.random() < 0.02
        }

        if not failure_flags["lost_pickup"] and not failure_flags["cancelled"]:
            # Pickup event
            pickup_time = generate_event_time(dispatch_time, 1, 72, failure_flags["delay_pickup"])
            pickup_note = "Late pickup due to weather" if failure_flags["delay_pickup"] else None
            pickup_metadata = {"DowntimeHours": round(random.uniform(0, 4),2)} if failure_flags["delay_pickup"] else None
            
            pickup_xml = create_shipping_xml_event(
                order_number, src, dst, items, weight, provider,
                "PickedUp", pickup_time, failure_note=pickup_note, metadata=pickup_metadata
            )
            upload_xml_to_blob(pickup_xml, f"{provider}_order_{order_number}_pickedup.xml")
            events_emitted += 1
            time.sleep(event_interval)

            # Routing event
            routing_delay_flag = failure_flags["delay_routing"] or failure_flags["reroute"]
            routing_time = generate_event_time(pickup_time, 1, 96, routing_delay_flag)

            routing_note = None
            if failure_flags["reroute"]:
                routing_note = random.choice([
                    "Rerouted due to customs",
                    "Rerouted due to logistics issue",
                    "Rerouted due to weather"
                ])
            elif failure_flags["delay_routing"]:
                routing_note = "Delayed in transit"

            routing_metadata = {}
            if failure_flags["reroute"]:
                routing_metadata["RerouteCount"] = random.randint(1,3)
            if failure_flags["delay_routing"]:
                routing_metadata["DelayHours"] = round(random.uniform(1, 24),2)

            routing_xml = create_shipping_xml_event(
                order_number, src, dst, items, weight, provider,
                "Routing", routing_time, failure_note=routing_note, metadata=routing_metadata if routing_metadata else None
            )
            upload_xml_to_blob(routing_xml, f"{provider}_order_{order_number}_routing.xml")
            events_emitted += 1
            time.sleep(event_interval)

            # Delivery or cancellation
            if failure_flags["cancelled"]:
                cancel_time = routing_time + timedelta(hours=random.uniform(1, 24))
                cancel_xml = create_shipping_xml_event(
                    order_number, src, dst, items, weight, provider,
                    "Cancelled", cancel_time,
                    failure_note="Shipment cancelled due to customer request",
                    metadata={"CancelReason": "CustomerRequest"}
                )
                upload_xml_to_blob(cancel_xml, f"{provider}_order_{order_number}_cancelled.xml")
                events_emitted += 1
                time.sleep(event_interval)
            else:
                delivered_time = routing_time + timedelta(hours=random.uniform(12, 72))
                delivered_xml = create_shipping_xml_event(
                    order_number, src, dst, items, weight, provider,
                    "Delivered", delivered_time
                )
                upload_xml_to_blob(delivered_xml, f"{provider}_order_{order_number}_delivered.xml")
                events_emitted += 1
                time.sleep(event_interval)

        else:
            # Cancelled or lost pickup immediately after dispatch
            if failure_flags["cancelled"]:
                cancel_time = dispatch_time + timedelta(hours=random.uniform(1, 12))
                cancel_xml = create_shipping_xml_event(
                    order_number, src, dst, items, weight, provider,
                    "Cancelled", cancel_time,
                    failure_note="Shipment cancelled before pickup",
                    metadata={"CancelReason": "PrePickup"}
                )
                upload_xml_to_blob(cancel_xml, f"{provider}_order_{order_number}_cancelled.xml")
                events_emitted += 1
                time.sleep(event_interval)
            # else the package is considered lost — no further events emitted

    print(f"Total uploaded shipment events: {events_emitted}")

In [ ]:
# Initialize Azure Blob Service client
blob_service_client = BlobServiceClient.from_connection_string(BLOB_CONNECTION_STRING)
container_client = blob_service_client.get_container_client(CONTAINER_NAME)
try:
    container_client.create_container()
except Exception:
    pass  # Container may already exist

In [ ]:
# Cell 3: Run the extended simulator for large dataset
generate_large_scale_shipping_events(total_orders=10000, max_events_per_second=50)